# 08_production_agent_system: Async Execution and Tracing Instrumentation

This notebook executes an async agent wrapper utilizing asyncio tasks, handling fallback routing and mock circuit-breaker configurations.


In [1]:
import asyncio

# 1. Async task execution wrapper
async def call_primary_endpoint(query: str):
    # Simulate transient api failure
    await asyncio.sleep(0.1)
    raise ConnectionError("Primary Model Provider rate limit hit (429).")

async def call_fallback_endpoint(query: str):
    # Fallback model execution
    await asyncio.sleep(0.1)
    return f"Fallback Model Response: Handled query '{query}' successfully."

async def execute_resilient_agent(query: str):
    print(f"Initiating request: '{query}'")
    try:
        print("Calling primary endpoint...")
        res = await call_primary_endpoint(query)
        return res
    except (ConnectionError, Exception) as e:
        print(f"Resilience Check: {e} Swapping to fallback endpoint...")
        res = await call_fallback_endpoint(query)
        return res

# Execute inside the notebook event loop
response = await execute_resilient_agent("Query user information")
print("\n" + response)


Initiating request: 'Query user information'
Calling primary endpoint...
Resilience Check: Primary Model Provider rate limit hit (429). Swapping to fallback endpoint...



Fallback Model Response: Handled query 'Query user information' successfully.


### Output Explanation
- The async runner manages API lifecycle events.
- If the primary provider returns a rate limit exception (429), the exception is caught, and the agent falls back to the secondary endpoint immediately.
